In [ ]:
import pandas as pd

df_full = pd.read_csv('df_with_marking_final_full.csv')

In [3]:
df_marking = df_full.sample(n=300, random_state=42).reset_index(drop=True)

In [ ]:
import re

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

male_forms = ["потерпевший", "потерпевшему", "потерпевшего", "потерпевшем"]
male_pattern = r"\b(" + "|".join(male_forms) + r")\b"

train_data_has_man_victim = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    has_man = row["predicted_has_man_victim"]
    if pd.isnull(has_man):
        continue

    if has_man == 1:
        match = re.search(male_pattern, text_lower)
        if match:
            start, end = match.span()
            train_data_has_man_victim.append((text, {"entities": [(start, end, "HAS_MAN_VICTIM")]}))
            # print(text[start:end])
        else:
            print(f"Не найдена форма 'потерпевший' при has_man_victim = 1 в id={row['id']}")
    else:
        train_data_has_man_victim.append((text, {"entities": []}))
        pass

print(f"TRAIN_DATA_HAS_MAN_VICTIM готово: {len(train_data_has_man_victim)} примеров")

Не найдена форма 'потерпевший' при has_man_victim = 1 в id=108974
Не найдена форма 'потерпевший' при has_man_victim = 1 в id=6041
Не найдена форма 'потерпевший' при has_man_victim = 1 в id=1415
TRAIN_DATA_HAS_MAN_VICTIM готово: 297 примеров


In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("HAS_MAN_VICTIM")

examples = []
for text, annot in train_data_has_man_victim:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(10):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_b_has_man_victim_model")
print("Модель сохранена в 'ner_b_has_man_victim_model'")

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Дело № 1-253/2016
ПРИГОВОР
Именем Российской Федер..." with entities "[(595, 607, 'HAS_MAN_VICTIM')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Дело № 1-116/2017
ПРИГОВОР
ИМЕНЕМ РОССИЙСКОЙ ФЕДЕР..." with entities "[(500, 512, 'HAS_MAN_VICTIM')]". 

Epoch 1, Losses: {'ner': 252565.16727562045}
Epoch 2, Losses: {'ner': 442.6061482845231}
Epoch 3, Losses: {'ner': 344.0694906034278}
Epoch 4, Losses: {'ner': 306.2937648922712}
Epoch 5, Losses: {'ner': 294.0335525715208}
Epoch 6, Losses: {'ner': 278.867676080808}
Epoch 7, Losses: {'ner': 244.92187920024975}
Epoch 8, Losses: {'ner': 222.4843579313597}
Epoch 9, Losses: {'ner': 211.38243317041002}
Epoch 10, Losses: {'ner': 196.3827863870259}
Модель сохранена в 'ner_b_has_man_victim_model'


In [8]:
df_test = pd.read_csv('df_with_marking_final.csv')

In [9]:
from sklearn.metrics import accuracy_score, f1_score

nlp = spacy.load("ner_b_has_man_victim_model")

y_true = []
y_pred = []

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

for _, row in df_test.iterrows():
    true_val = row["has_man_victim"]
    if pd.isnull(true_val):
        continue

    text = get_full_text(row)
    doc = nlp(text.lower())

    predicted_val = 0
    for ent in doc.ents:
        if ent.label_ == "HAS_MAN_VICTIM":
            print(ent.text)
            predicted_val = 1
            break

    y_true.append(int(true_val))
    y_pred.append(predicted_val)

# Метрики
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.2%}")
print(f"F1-score: {f1:.2%}")

потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшему
потерпевшего
потерпевший
потерпевшего
потерпевшему
потерпевший
потерпевший
потерпевшего
потерпевшего
потерпевшему
потерпевший
потерпевший
потерпевшего
потерпевшего
потерпевший
потерпевшему
потерпевшего
потерпевший
потерпевшему
потерпевшего
потерпевший
потерпевшему
потерпевшего
потерпевшему
потерпевшего
потерпевшего
потерпевшему
Accuracy: 49.00%
F1-score: 50.90%
